# ARIA v4.0 — Clean Rebuild

This notebook is a cleaned and reorganized rebuild of the Week 7 ARIA v4.0 assignment.

## What this version fixes
1. Removes duplicated and outdated cells.
2. Keeps all spatial analysis in **EPSG:3826** and converts to **EPSG:4326** only for Folium.
3. Restricts facilities to the actual study network area.
4. Uses stable **nearest-node** facility points for isochrone plotting.
5. Produces cleaner outputs:
   - `betweenness_centrality.csv`
   - `top5_bottlenecks.csv`
   - `bottleneck_terrain_overlay.csv`
   - `rainfall_stations_filtered.geojson`
   - `road_edge_midpoints_with_rainfall.geojson`
   - `accessibility_impact_table.csv`
   - `isochrone_detailed_results.csv`
   - `isochrone_*.png`
   - `accessibility_summary_plots.png`
   - `accessibility_interactive_map.html`
   - `README.md`

## Step 0 — Setup and Imports

This cell imports packages, configures fonts, and defines reusable helpers.

In [ ]:
from __future__ import annotations

import os
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import folium
import matplotlib.pyplot as plt
import matplotlib as mpl

from shapely.geometry import Point, box
from shapely import wkt
from matplotlib import font_manager

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# -----------------------------
# Chinese font setup
# -----------------------------
candidate_fonts = [
    "Microsoft JhengHei",
    "PingFang TC",
    "Heiti TC",
    "Noto Sans CJK TC",
    "Noto Sans CJK JP",
    "SimHei",
    "Arial Unicode MS",
]
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
chosen_font = next((f for f in candidate_fonts if f in available_fonts), None)

if chosen_font:
    mpl.rcParams["font.family"] = chosen_font
    print(f"✅ Using Chinese font: {chosen_font}")
else:
    print("⚠️ No common Chinese font found. Chinese text may not render correctly.")

mpl.rcParams["axes.unicode_minus"] = False

# -----------------------------
# Reusable helpers
# -----------------------------
def safe_filename(name: str) -> str:
    return str(name).replace("/", "_").replace("\\", "_").replace(" ", "_")

def infer_point_crs(x: pd.Series, y: pd.Series) -> str:
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    if x.between(119, 123).all() and y.between(21, 26).all():
        return "EPSG:4326"
    if x.mean() > 1000 and y.mean() > 1000:
        return "EPSG:3826"
    raise ValueError(
        f"Unable to infer CRS from coordinate ranges: "
        f"x=({x.min()}, {x.max()}), y=({y.min()}, {y.max()})"
    )

def parse_speed(maxspeed_attr, highway_attr, default_speed=40.0):
    speed_map = {
        "motorway": 90,
        "trunk": 70,
        "primary": 60,
        "secondary": 50,
        "tertiary": 40,
        "residential": 30,
        "service": 20,
        "unclassified": 30,
        "living_street": 20,
    }

    if isinstance(maxspeed_attr, list) and len(maxspeed_attr) > 0:
        maxspeed_attr = maxspeed_attr[0]

    if isinstance(maxspeed_attr, str):
        s = maxspeed_attr.lower().replace("km/h", "").replace("kph", "").replace("mph", "").strip()
        if "-" in s:
            try:
                vals = [float(v.strip()) for v in s.split("-")]
                return float(np.mean(vals))
            except Exception:
                pass
        try:
            return float(s)
        except Exception:
            pass

    if isinstance(maxspeed_attr, (int, float)) and not pd.isna(maxspeed_attr):
        return float(maxspeed_attr)

    if isinstance(highway_attr, list) and len(highway_attr) > 0:
        highway_attr = highway_attr[0]

    if isinstance(highway_attr, str):
        return float(speed_map.get(highway_attr, default_speed))

    return float(default_speed)

def rain_to_congestion(rain_mm: float, light=15, moderate=30, heavy=50, extreme=80) -> float:
    if pd.isna(rain_mm) or rain_mm <= 0:
        return 0.0
    if rain_mm <= light:
        return 0.10
    if rain_mm <= moderate:
        return 0.30
    if rain_mm <= heavy:
        return 0.60
    if rain_mm <= extreme:
        return 0.85
    return 0.95

def adjusted_travel_time(original_time: float, cf: float) -> float:
    if pd.isna(cf):
        cf = 0.0
    if cf >= 0.95:
        return np.inf
    return float(original_time) / (1 - float(cf))

def congestion_label(cf: float) -> str:
    if pd.isna(cf) or cf == 0:
        return "No Impact"
    if cf <= 0.10:
        return "Light Impact"
    if cf <= 0.30:
        return "Moderate Impact"
    if cf <= 0.60:
        return "Heavy Impact"
    if cf < 0.95:
        return "Extreme Impact"
    return "Critical Impact"

def polygon_area_km2(geom) -> float:
    if geom is None or geom.is_empty:
        return 0.0
    return float(gpd.GeoSeries([geom], crs="EPSG:3826").area.iloc[0] / 1_000_000)

def make_isochrone_polygon(graph, source_node, weight_col, time_minutes, gdf_nodes, node_buffer=80):
    cutoff_sec = float(time_minutes) * 60
    try:
        lengths = nx.single_source_dijkstra_path_length(
            graph,
            source=source_node,
            cutoff=cutoff_sec,
            weight=weight_col
        )
    except Exception:
        return None, [], True

    reachable_nodes = list(lengths.keys())
    if len(reachable_nodes) == 0:
        return None, [], True

    node_points = gdf_nodes.loc[reachable_nodes].geometry.tolist()
    buffered = [pt.buffer(node_buffer) for pt in node_points]
    union_geom = gpd.GeoSeries(buffered, crs="EPSG:3826").unary_union

    # Robust default: convex hull of reachable node buffers
    poly = union_geom.convex_hull
    if poly is None or poly.is_empty:
        return None, reachable_nodes, False

    return poly, reachable_nodes, False

def get_local_bounds(pre_poly, post_poly, facility_point, pad=1000):
    geoms = []
    if pre_poly is not None and not pre_poly.is_empty:
        geoms.append(pre_poly)
    if post_poly is not None and not post_poly.is_empty:
        geoms.append(post_poly)
    if facility_point is not None and not facility_point.is_empty:
        geoms.append(facility_point)

    if len(geoms) == 0:
        return None

    union_geom = gpd.GeoSeries(geoms, crs="EPSG:3826").unary_union
    minx, miny, maxx, maxy = union_geom.bounds
    width = maxx - minx
    height = maxy - miny

    if width < 1200:
        midx = (minx + maxx) / 2
        minx, maxx = midx - 600, midx + 600
    if height < 1200:
        midy = (miny + maxy) / 2
        miny, maxy = midy - 600, midy + 600

    return (minx - pad, miny - pad, maxx + pad, maxy + pad)

def simplify_for_web(geom, tolerance=0.00012):
    if geom is None or geom.is_empty:
        return None
    try:
        return geom.simplify(tolerance, preserve_topology=True)
    except Exception:
        return geom

print("✅ Imports and helpers ready")


## Step 1 — Configuration and Paths

Adjust these values if needed. The defaults are set to match the uploaded files in this conversation.

In [ ]:
BASE_DIR = Path(".").resolve()
OUTPUTS_DIR = BASE_DIR / "outputs_aria_v4_clean"
OUTPUTS_DIR.mkdir(exist_ok=True)

# Core inputs
NETWORK_GRAPHML = BASE_DIR / "hualien_network.graphml"
SHELTER_FILE = BASE_DIR / "shelters_composite_risk.csv"
RAINFALL_JSON = BASE_DIR / "fungwong_202511.json"

# Main configuration
TARGET_AREA = "Hualien City, Taiwan"
NETWORK_TYPE = "drive"
DEFAULT_SPEED = 40.0
CENTRALITY_NORMALIZE = True

# Rain-congestion thresholds
RAIN_LIGHT = 15.0
RAIN_MODERATE = 30.0
RAIN_HEAVY = 50.0
RAIN_EXTREME = 80.0
RAIN_SEARCH_RADIUS = 5000.0  # meters

print("=== Configuration ===")
print("TARGET_AREA:", TARGET_AREA)
print("NETWORK_TYPE:", NETWORK_TYPE)
print("DEFAULT_SPEED:", DEFAULT_SPEED)
print("NETWORK_GRAPHML:", NETWORK_GRAPHML)
print("SHELTER_FILE exists:", SHELTER_FILE.exists())
print("RAINFALL_JSON exists:", RAINFALL_JSON.exists())
print("OUTPUTS_DIR:", OUTPUTS_DIR)


## Step 2 — Road Network Extraction and Projection

This cell loads a saved GraphML if available; otherwise it fetches from OSM and saves a local copy.

In [ ]:
print("=== Captain's Log: Road Network Extraction ===")

G = None
if NETWORK_GRAPHML.exists():
    print(f"📁 Loading cached network: {NETWORK_GRAPHML}")
    G = ox.load_graphml(NETWORK_GRAPHML)

if G is None:
    print(f"🌐 Fetching {NETWORK_TYPE} network for {TARGET_AREA}")
    G = ox.graph_from_place(TARGET_AREA, network_type=NETWORK_TYPE)
    ox.save_graphml(G, NETWORK_GRAPHML)
    print(f"✅ Saved GraphML to {NETWORK_GRAPHML}")

# Project to EPSG:3826 for all meter-based work
G = ox.project_graph(G, to_crs="EPSG:3826")

gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
print(f"✅ Projected graph: {len(G.nodes)} nodes, {len(G.edges)} edges")
print("Graph CRS:", gdf_edges.crs)


## Step 3 — Travel Time Calculation

This cell extracts speed values and computes baseline travel time for every road segment.

In [ ]:
print("=== Captain's Log: Travel Time Calculation ===")

for u, v, k, data in G.edges(keys=True, data=True):
    speed_kmh = parse_speed(data.get("maxspeed"), data.get("highway"), DEFAULT_SPEED)
    length_m = float(data.get("length", 0.0))
    travel_time = np.inf if speed_kmh <= 0 else length_m / (speed_kmh / 3.6)

    data["speed_kmh"] = float(speed_kmh)
    data["travel_time"] = float(travel_time)

gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges = gdf_edges.copy()

print(gdf_edges[["length", "speed_kmh", "travel_time"]].head())
print("✅ Baseline travel time added to all edges")


## Step 4 — Betweenness Centrality and Top-5 Bottlenecks

In [ ]:
print("=== Captain's Log: Betweenness Centrality Analysis ===")

centrality = nx.betweenness_centrality(
    G,
    weight="travel_time",
    normalized=CENTRALITY_NORMALIZE
)

centrality_df = (
    pd.DataFrame({"node_id": list(centrality.keys()), "betweenness": list(centrality.values())})
    .sort_values("betweenness", ascending=False)
    .reset_index(drop=True)
)

top5_bottlenecks = centrality_df.head(5).copy()
top5_bottlenecks["rank"] = range(1, 6)

centrality_df.to_csv(OUTPUTS_DIR / "betweenness_centrality.csv", index=False)
top5_bottlenecks.to_csv(OUTPUTS_DIR / "top5_bottlenecks.csv", index=False)

print(top5_bottlenecks)
print("✅ Centrality outputs saved")


## Step 5 — Shelter Loading, CRS Repair, and Study-Area Filtering

The uploaded shelter CSV uses lon/lat in WGS84, so this cell:
1. detects coordinate columns and CRS,
2. reprojects to EPSG:3826,
3. keeps only shelters within the road-network study area,
4. snaps each shelter to its nearest network node.

In [ ]:
print("=== Captain's Log: Shelter Data Loading ===")

if not SHELTER_FILE.exists():
    raise FileNotFoundError(f"Missing shelter file: {SHELTER_FILE}")

shelters_df = pd.read_csv(SHELTER_FILE)
print(f"Loaded {len(shelters_df)} shelter rows")

# Identify name column
name_candidates = ["name", "避難收容處所名稱", "shelter_name"]
name_col = next((c for c in name_candidates if c in shelters_df.columns), None)
if name_col is None:
    raise ValueError("No shelter name column found")

coord_candidates = [
    ("longitude", "latitude"),
    ("經度", "緯度"),
    ("lon", "lat"),
    ("X", "Y"),
    ("x", "y"),
]
x_col, y_col = next(((xc, yc) for xc, yc in coord_candidates if xc in shelters_df.columns and yc in shelters_df.columns), (None, None))
if x_col is None:
    raise ValueError("No coordinate columns found")

input_crs = infer_point_crs(shelters_df[x_col], shelters_df[y_col])
print("Using columns:", x_col, y_col, "| inferred CRS:", input_crs)

shelters_gdf = gpd.GeoDataFrame(
    shelters_df.copy(),
    geometry=gpd.points_from_xy(
        pd.to_numeric(shelters_df[x_col], errors="coerce"),
        pd.to_numeric(shelters_df[y_col], errors="coerce")
    ),
    crs=input_crs
).dropna(subset=["geometry"]).to_crs("EPSG:3826")

# Study area = network convex hull + 5 km buffer
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
study_area = gdf_edges.unary_union.convex_hull.buffer(5000)
study_area_gdf = gpd.GeoDataFrame({"name": ["study_area"]}, geometry=[study_area], crs="EPSG:3826")

shelters_gdf = shelters_gdf[shelters_gdf.geometry.within(study_area)].copy()
print(f"Shelters inside study area: {len(shelters_gdf)}")

if len(shelters_gdf) == 0:
    raise ValueError("No shelters remain after study-area filtering.")

# Standardize core columns
if "shelter_id" not in shelters_gdf.columns:
    shelters_gdf["shelter_id"] = [f"SH{i:04d}" for i in range(1, len(shelters_gdf) + 1)]
if "name" not in shelters_gdf.columns:
    shelters_gdf["name"] = shelters_gdf[name_col]

# Snap to nearest node
shelters_gdf["nearest_node"] = shelters_gdf.geometry.apply(
    lambda geom: ox.distance.nearest_nodes(G, X=geom.x, Y=geom.y)
)

# Remove duplicate snapped nodes
before = len(shelters_gdf)
shelters_gdf = shelters_gdf.drop_duplicates(subset="nearest_node").copy()
after = len(shelters_gdf)

print(f"Removed duplicated nearest-node shelters: {before - after}")
display_cols = [c for c in ["shelter_id", "name", "縣市及鄉鎮市區", "terrain_risk", "nearest_node"] if c in shelters_gdf.columns]
print(shelters_gdf[display_cols].head(10).to_string(index=False))

shelters_gdf.to_file(OUTPUTS_DIR / "shelters_filtered_to_network.geojson", driver="GeoJSON")
study_area_gdf.to_file(OUTPUTS_DIR / "study_area.geojson", driver="GeoJSON")
print("✅ Shelter outputs saved")


## Step 6 — Top-5 Bottleneck Terrain Overlay

The assignment asks for Top-5 bottlenecks overlaid with Week 4 terrain risk.  
This version approximates terrain context by attaching each bottleneck node to its **nearest shelter record** and borrowing the shelter's `terrain_risk`, `mean_elevation`, and `max_slope`.

In [ ]:
print("=== Captain's Log: Bottleneck Terrain Overlay ===")

top5_nodes = gdf_nodes.loc[top5_bottlenecks["node_id"]].copy()
top5_nodes["node_id"] = top5_nodes.index
top5_nodes = top5_nodes[["node_id", "geometry"]].copy()

top5_nodes_gdf = gpd.GeoDataFrame(top5_nodes, geometry="geometry", crs="EPSG:3826")

nearest_cols = [c for c in ["shelter_id", "name", "terrain_risk", "mean_elevation", "max_slope", "geometry"] if c in shelters_gdf.columns]
bottleneck_terrain_df = gpd.sjoin_nearest(
    top5_nodes_gdf,
    shelters_gdf[nearest_cols],
    how="left",
    distance_col="distance_to_shelter"
).drop(columns=["index_right"], errors="ignore")

bottleneck_terrain_df = bottleneck_terrain_df.merge(
    top5_bottlenecks[["node_id", "rank", "betweenness"]],
    on="node_id",
    how="left"
).sort_values("rank")

# Friendly output column names
if "name" in bottleneck_terrain_df.columns:
    bottleneck_terrain_df = bottleneck_terrain_df.rename(columns={"name": "nearest_shelter"})
if "mean_elevation" in bottleneck_terrain_df.columns:
    bottleneck_terrain_df = bottleneck_terrain_df.rename(columns={"mean_elevation": "mean_elev"})

save_cols = [c for c in ["rank", "node_id", "betweenness", "nearest_shelter", "terrain_risk", "mean_elev", "max_slope", "distance_to_shelter"] if c in bottleneck_terrain_df.columns]
bottleneck_terrain_df[save_cols].to_csv(OUTPUTS_DIR / "bottleneck_terrain_overlay.csv", index=False)

print(bottleneck_terrain_df[save_cols].to_string(index=False))
print("✅ Bottleneck terrain overlay saved")


## Step 7 — Rainfall Parsing and Network Integration

This cell parses the uploaded CWA rainfall JSON.  
For dynamic congestion, it uses **Past1hr precipitation** because the assignment is framed around rainfall-driven travel delay.

In [ ]:
print("=== Captain's Log: Rainfall Data Integration ===")

if not RAINFALL_JSON.exists():
    raise FileNotFoundError(f"Missing rainfall JSON: {RAINFALL_JSON}")

with open(RAINFALL_JSON, "r", encoding="utf-8") as f:
    rainfall_data = json.load(f)

if "records" not in rainfall_data or "Station" not in rainfall_data["records"]:
    raise ValueError("Unexpected rainfall JSON structure. Expected records -> Station")

station_list = rainfall_data["records"]["Station"]
rows = []

for st in station_list:
    try:
        sid = st.get("StationId", "UNKNOWN")
        sname = st.get("StationName", "UNKNOWN")
        coords = st["GeoInfo"]["Coordinates"][0]
        lon = float(coords["StationLongitude"])
        lat = float(coords["StationLatitude"])
        rain_1h = float(st["RainfallElement"]["Past1hr"]["Precipitation"])
        if rain_1h < -900:
            continue

        county = st.get("GeoInfo", {}).get("CountyName")
        town = st.get("GeoInfo", {}).get("TownName")
        rows.append({
            "station_id": sid,
            "station_name": sname,
            "county": county,
            "town": town,
            "lon": lon,
            "lat": lat,
            "rainfall_mm": rain_1h
        })
    except Exception:
        continue

rainfall_df = pd.DataFrame(rows)
rainfall_gdf = gpd.GeoDataFrame(
    rainfall_df,
    geometry=gpd.points_from_xy(rainfall_df["lon"], rainfall_df["lat"]),
    crs="EPSG:4326"
).to_crs("EPSG:3826")

# Keep only stations near the study network
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
rain_station_buffer = gdf_edges.unary_union.convex_hull.buffer(30000)  # 30 km
rainfall_gdf = rainfall_gdf[rainfall_gdf.geometry.within(rain_station_buffer)].copy()

if len(rainfall_gdf) == 0:
    raise ValueError("No rainfall stations remain near the study area.")

rainfall_gdf["congestion_factor"] = rainfall_gdf["rainfall_mm"].apply(
    lambda x: rain_to_congestion(x, RAIN_LIGHT, RAIN_MODERATE, RAIN_HEAVY, RAIN_EXTREME)
)

print(rainfall_gdf[["station_id", "station_name", "county", "town", "rainfall_mm", "congestion_factor"]].head(15).to_string(index=False))

# Edge midpoints
edges_reset = gdf_edges.reset_index().copy()
edges_reset["midpoint"] = edges_reset.geometry.interpolate(0.5, normalized=True)

edge_mid_gdf = gpd.GeoDataFrame(
    edges_reset[["u", "v", "key", "length", "speed_kmh", "travel_time"]].copy(),
    geometry=edges_reset["midpoint"],
    crs="EPSG:3826"
)

edge_rainfall = gpd.sjoin_nearest(
    edge_mid_gdf,
    rainfall_gdf[["station_id", "station_name", "rainfall_mm", "congestion_factor", "geometry"]],
    how="left",
    distance_col="distance_to_station"
).drop(columns=["index_right"], errors="ignore")

median_rain = rainfall_gdf["rainfall_mm"].median()
median_cf = rain_to_congestion(median_rain, RAIN_LIGHT, RAIN_MODERATE, RAIN_HEAVY, RAIN_EXTREME)

far_mask = edge_rainfall["distance_to_station"] > RAIN_SEARCH_RADIUS
if far_mask.any():
    edge_rainfall.loc[far_mask, "station_id"] = "FAR_DEFAULT"
    edge_rainfall.loc[far_mask, "station_name"] = "FarFromStation_Default"
    edge_rainfall.loc[far_mask, "rainfall_mm"] = median_rain
    edge_rainfall.loc[far_mask, "congestion_factor"] = median_cf

edge_rainfall["travel_time_adj"] = edge_rainfall.apply(
    lambda row: adjusted_travel_time(row["travel_time"], row["congestion_factor"]),
    axis=1
)
edge_rainfall["congestion_category"] = edge_rainfall["congestion_factor"].apply(congestion_label)

# Write adjusted weights back to graph
for _, row in edge_rainfall.iterrows():
    u, v, key = row["u"], row["v"], row["key"]
    if G.has_edge(u, v, key):
        G[u][v][key]["rainfall_mm"] = float(row["rainfall_mm"])
        G[u][v][key]["congestion_factor"] = float(row["congestion_factor"])
        G[u][v][key]["travel_time_adj"] = float(row["travel_time_adj"]) if np.isfinite(row["travel_time_adj"]) else np.inf
        G[u][v][key]["congestion_category"] = row["congestion_category"]
        G[u][v][key]["distance_to_station"] = float(row["distance_to_station"])

rainfall_gdf.to_file(OUTPUTS_DIR / "rainfall_stations_filtered.geojson", driver="GeoJSON")
edge_rainfall.rename_geometry("midpoint_geometry").to_file(
    OUTPUTS_DIR / "road_edge_midpoints_with_rainfall.geojson",
    driver="GeoJSON"
)

congestion_counts = edge_rainfall["congestion_category"].value_counts()
print(congestion_counts.to_string())
print("✅ Rainfall-network integration saved")


## Step 8 — Key Facility Selection and Isochrone Analysis

This cell:
1. selects five facilities that remain inside the study area,
2. computes pre-disaster and post-disaster 5-minute / 10-minute isochrones,
3. calculates area shrinkage.

In [ ]:
print("=== Captain's Log: Isochrone Analysis ===")

if len(shelters_gdf) < 5:
    raise ValueError("Fewer than 5 shelters remain after study-area filtering.")

# Prefer some terrain-risk diversity, then fill remaining slots
selected = []

if "terrain_risk" in shelters_gdf.columns:
    for risk_level in ["HIGH", "MEDIUM", "LOW"]:
        subset = shelters_gdf[shelters_gdf["terrain_risk"].astype(str).str.upper() == risk_level]
        if len(subset) > 0:
            selected.append(subset.head(2))

selected_facilities = pd.concat(selected, axis=0) if len(selected) > 0 else shelters_gdf.copy()
selected_facilities = selected_facilities.drop_duplicates(subset="nearest_node").head(5).copy()

if len(selected_facilities) < 5:
    remain = shelters_gdf.loc[~shelters_gdf.index.isin(selected_facilities.index)]
    remain = remain.drop_duplicates(subset="nearest_node")
    selected_facilities = pd.concat([selected_facilities, remain.head(5 - len(selected_facilities))], axis=0)

selected_facilities = selected_facilities.head(5).copy()

print("Selected facilities:")
print(selected_facilities[[c for c in ["shelter_id", "name", "縣市及鄉鎮市區", "terrain_risk", "nearest_node"] if c in selected_facilities.columns]].to_string(index=False))

# Build post-disaster graph with impassable edges removed
G_post = G.copy()
impassable_edges = []
for u, v, k, data in G_post.edges(keys=True, data=True):
    if data.get("congestion_factor", 0) >= 0.95 or not np.isfinite(data.get("travel_time_adj", np.inf)):
        impassable_edges.append((u, v, k))
if impassable_edges:
    G_post.remove_edges_from(impassable_edges)

gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)

time_limits = [5, 10]
isochrone_results = []
isolated_facilities = []

for _, facility in selected_facilities.iterrows():
    facility_id = facility["shelter_id"]
    facility_name = facility["name"]
    source_node = facility["nearest_node"]

    for tmin in time_limits:
        pre_poly, pre_nodes, _ = make_isochrone_polygon(G, source_node, "travel_time", tmin, gdf_nodes)
        post_poly, post_nodes, _ = make_isochrone_polygon(G_post, source_node, "travel_time_adj", tmin, gdf_nodes)

        pre_area = polygon_area_km2(pre_poly)
        post_area = polygon_area_km2(post_poly)
        shrinkage = 0.0 if pre_area == 0 else (1 - post_area / pre_area) * 100

        isochrone_results.append({
            "facility_id": facility_id,
            "facility_name": facility_name,
            "time_minutes": tmin,
            "pre_polygon": pre_poly,
            "post_polygon": post_poly,
            "pre_disaster_area_km2": pre_area,
            "post_disaster_area_km2": post_area,
            "shrinkage_percent": shrinkage,
            "pre_reachable_nodes": len(pre_nodes),
            "post_reachable_nodes": len(post_nodes),
            "isolated_post_disaster": len(post_nodes) <= 1
        })

    # mark isolation using 10-minute post state
    ten_min_result = next(r for r in isochrone_results if r["facility_name"] == facility_name and r["time_minutes"] == 10)
    if ten_min_result["isolated_post_disaster"]:
        isolated_facilities.append(facility_name)

# Build final table
rows = []
for fname in selected_facilities["name"]:
    r5 = next(r for r in isochrone_results if r["facility_name"] == fname and r["time_minutes"] == 5)
    r10 = next(r for r in isochrone_results if r["facility_name"] == fname and r["time_minutes"] == 10)
    rows.append({
        "Facility": fname,
        "Pre-Disaster 5min (km²)": r5["pre_disaster_area_km2"],
        "Post-Disaster 5min (km²)": r5["post_disaster_area_km2"],
        "Shrinkage % (5min)": r5["shrinkage_percent"],
        "Pre-Disaster 10min (km²)": r10["pre_disaster_area_km2"],
        "Post-Disaster 10min (km²)": r10["post_disaster_area_km2"],
        "Shrinkage % (10min)": r10["shrinkage_percent"],
    })

accessibility_table = pd.DataFrame(rows)
accessibility_table.to_csv(OUTPUTS_DIR / "accessibility_impact_table.csv", index=False)

detail_df = pd.DataFrame([
    {
        "facility_id": r["facility_id"],
        "facility_name": r["facility_name"],
        "time_minutes": r["time_minutes"],
        "pre_disaster_area_km2": r["pre_disaster_area_km2"],
        "post_disaster_area_km2": r["post_disaster_area_km2"],
        "shrinkage_percent": r["shrinkage_percent"],
        "pre_reachable_nodes": r["pre_reachable_nodes"],
        "post_reachable_nodes": r["post_reachable_nodes"],
        "isolated_post_disaster": r["isolated_post_disaster"],
    }
    for r in isochrone_results
])
detail_df.to_csv(OUTPUTS_DIR / "isochrone_detailed_results.csv", index=False)

print(accessibility_table.to_string(index=False))
print("Isolated facilities:", isolated_facilities if isolated_facilities else "None")
print("✅ Isochrone analysis outputs saved")


## Step 9 — Clean Visualization Outputs

This cell produces:
- one PNG per facility,
- one summary comparison PNG,
- one interactive Folium HTML map.

### Visualization improvements
- uses local map windows for each facility,
- avoids plotting the full road network in every panel,
- keeps labels inside the frame,
- groups Folium polygons by facility to reduce clutter.

In [ ]:
print("=== Captain's Log: Visualization ===")

gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_nodes_wgs84 = gdf_nodes.to_crs("EPSG:4326")
gdf_edges_wgs84 = gdf_edges.to_crs("EPSG:4326")
selected_facilities_wgs84 = selected_facilities.to_crs("EPSG:4326")
node_geom_lookup = gdf_nodes.geometry.to_dict()

result_lookup = {(r["facility_name"], r["time_minutes"]): r for r in isochrone_results}

# -----------------------------
# Facility-specific static plots
# -----------------------------
for _, facility in selected_facilities.iterrows():
    facility_name = facility["name"]
    facility_node = facility["nearest_node"]
    facility_point = node_geom_lookup[facility_node]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

    for ax, time_minutes in zip(axes, [5, 10]):
        result = result_lookup.get((facility_name, time_minutes))
        if result is None:
            ax.text(0.5, 0.5, "無資料", ha="center", va="center", transform=ax.transAxes)
            continue

        pre_poly = result["pre_polygon"]
        post_poly = result["post_polygon"]

        bounds = get_local_bounds(pre_poly, post_poly, facility_point, pad=1000)
        if bounds is None:
            ax.text(0.5, 0.5, "無幾何資料", ha="center", va="center", transform=ax.transAxes)
            continue

        minx, miny, maxx, maxy = bounds
        bbox = box(minx, miny, maxx, maxy)

        local_edges = gdf_edges[gdf_edges.intersects(bbox)].copy()
        if len(local_edges) > 0:
            local_edges.plot(ax=ax, linewidth=0.8, color="lightgray", alpha=0.9, zorder=1)

        polys = []
        if pre_poly is not None and not pre_poly.is_empty:
            polys.append(("災前", pre_poly, "blue", 0.28, 2))
        if post_poly is not None and not post_poly.is_empty:
            polys.append(("災後", post_poly, "red", 0.28, 3))
        polys = sorted(polys, key=lambda x: x[1].area if x[1] is not None else 0, reverse=True)

        for label, poly, color, alpha, z in polys:
            gpd.GeoSeries([poly], crs="EPSG:3826").plot(
                ax=ax, color=color, alpha=alpha, edgecolor=color, linewidth=2, zorder=z, label=label
            )

        gpd.GeoSeries([facility_point], crs="EPSG:3826").plot(
            ax=ax, color="black", marker="*", markersize=180, zorder=5, label="設施位置"
        )

        info_text = (
            f"災前: {result['pre_disaster_area_km2']:.3f} km²\n"
            f"災後: {result['post_disaster_area_km2']:.3f} km²\n"
            f"縮減: {result['shrinkage_percent']:.1f}%"
        )
        ax.text(
            0.02, 0.98, info_text,
            transform=ax.transAxes,
            fontsize=10,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9)
        )

        ax.set_title(f"{facility_name}\n{time_minutes}分鐘可達範圍", fontsize=15, fontweight="bold")
        ax.set_xlabel("Easting (m)")
        ax.set_ylabel("Northing (m)")
        ax.set_xlim(minx, maxx)
        ax.set_ylim(miny, maxy)
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.25)
        ax.legend(loc="lower left", framealpha=0.95)

    fig.savefig(OUTPUTS_DIR / f"isochrone_{safe_filename(facility_name)}.png", dpi=250, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# -----------------------------
# Summary plots
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)

facility_names = [r["facility_name"] for r in isochrone_results if r["time_minutes"] == 5]
shrinkage_5min = [r["shrinkage_percent"] for r in isochrone_results if r["time_minutes"] == 5]
shrinkage_10min = [r["shrinkage_percent"] for r in isochrone_results if r["time_minutes"] == 10]
pre_5min = [r["pre_disaster_area_km2"] for r in isochrone_results if r["time_minutes"] == 5]
post_5min = [r["post_disaster_area_km2"] for r in isochrone_results if r["time_minutes"] == 5]

x_pos = np.arange(len(facility_names))
width = 0.35

ax1 = axes[0, 0]
ax1.bar(x_pos - width / 2, shrinkage_5min, width, label="5分鐘", alpha=0.85)
ax1.bar(x_pos + width / 2, shrinkage_10min, width, label="10分鐘", alpha=0.85)
ax1.set_xlabel("設施")
ax1.set_ylabel("可達性縮減 (%)")
ax1.set_title("各設施可達性縮減比較", fontweight="bold")
ax1.set_xticks(x_pos)
ax1.set_xticklabels([n[:10] + "…" if len(n) > 10 else n for n in facility_names], rotation=35, ha="right")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.bar(x_pos - width / 2, pre_5min, width, label="災前", alpha=0.85)
ax2.bar(x_pos + width / 2, post_5min, width, label="災後", alpha=0.85)
ax2.set_xlabel("設施")
ax2.set_ylabel("可達面積 (km²)")
ax2.set_title("5分鐘可達面積比較", fontweight="bold")
ax2.set_xticks(x_pos)
ax2.set_xticklabels([n[:10] + "…" if len(n) > 10 else n for n in facility_names], rotation=35, ha="right")
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
all_shrinkage = [r["shrinkage_percent"] for r in isochrone_results]
ax3.hist(all_shrinkage, bins=10, alpha=0.75, edgecolor="black")
ax3.axvline(np.mean(all_shrinkage), linestyle="--", linewidth=2, color="red", label=f"平均: {np.mean(all_shrinkage):.1f}%")
ax3.set_xlabel("可達性縮減 (%)")
ax3.set_ylabel("頻率")
ax3.set_title("可達性縮減分布", fontweight="bold")
ax3.grid(True, alpha=0.3)
ax3.legend()

ax4 = axes[1, 1]
risk_impact_rows = []
for fname in selected_facilities["name"]:
    fdata = selected_facilities[selected_facilities["name"] == fname].iloc[0]
    fr = [r for r in isochrone_results if r["facility_name"] == fname]
    if len(fr) > 0:
        risk_impact_rows.append({
            "facility": fname,
            "risk": fdata.get("terrain_risk", "UNKNOWN"),
            "shrinkage": float(np.mean([r["shrinkage_percent"] for r in fr]))
        })

risk_impact_df = pd.DataFrame(risk_impact_rows)
if len(risk_impact_df) > 0:
    grouped = risk_impact_df.groupby("risk")["shrinkage"].agg(["mean", "std"]).reset_index()
    for _, row in grouped.iterrows():
        yerr_val = 0 if pd.isna(row["std"]) else row["std"]
        ax4.bar(row["risk"], row["mean"], yerr=yerr_val, alpha=0.85, capsize=5)
    ax4.set_xlabel("地形風險等級")
    ax4.set_ylabel("平均可達性縮減 (%)")
    ax4.set_title("地形風險與可達性衝擊", fontweight="bold")
    ax4.grid(True, alpha=0.3)
else:
    ax4.text(0.5, 0.5, "無資料", ha="center", va="center", transform=ax4.transAxes)

fig.savefig(OUTPUTS_DIR / "accessibility_summary_plots.png", dpi=250, bbox_inches="tight")
plt.show()
plt.close(fig)

# -----------------------------
# Interactive Folium map
# -----------------------------
center_lat = gdf_nodes_wgs84.geometry.y.mean()
center_lon = gdf_nodes_wgs84.geometry.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="OpenStreetMap")

roads_fg = folium.FeatureGroup(name="Road Network", show=True)
sample_edges = gdf_edges_wgs84.sample(min(1200, len(gdf_edges_wgs84)), random_state=42) if len(gdf_edges_wgs84) > 1200 else gdf_edges_wgs84
for _, edge in sample_edges.iterrows():
    try:
        coords = [(y, x) for x, y in edge.geometry.coords]
        folium.PolyLine(coords, color="gray", weight=1, opacity=0.45).add_to(roads_fg)
    except Exception:
        pass
roads_fg.add_to(m)

facility_fg = folium.FeatureGroup(name="Facilities", show=True)
for _, facility in selected_facilities_wgs84.iterrows():
    popup_text = f"<b>{facility['name']}</b><br>ID: {facility['shelter_id']}"
    if "terrain_risk" in facility.index:
        popup_text += f"<br>Terrain Risk: {facility['terrain_risk']}"
    if "mean_elevation" in facility.index and pd.notna(facility["mean_elevation"]):
        popup_text += f"<br>Elevation: {facility['mean_elevation']:.1f} m"

    folium.Marker(
        location=[facility.geometry.y, facility.geometry.x],
        popup=folium.Popup(popup_text, max_width=280),
        tooltip=facility["name"],
        icon=folium.Icon(color="red", icon="home", prefix="fa")
    ).add_to(facility_fg)
facility_fg.add_to(m)

for facility_name in selected_facilities["name"]:
    facility_group = folium.FeatureGroup(name=f"{facility_name} Isochrones", show=False)

    for time_minutes in [5, 10]:
        result = result_lookup.get((facility_name, time_minutes))
        if result is None:
            continue

        if result["pre_polygon"] is not None and not result["pre_polygon"].is_empty:
            pre_wgs84 = gpd.GeoSeries([result["pre_polygon"]], crs="EPSG:3826").to_crs("EPSG:4326").iloc[0]
            pre_wgs84 = simplify_for_web(pre_wgs84, tolerance=0.00012)
            if pre_wgs84 is not None and not pre_wgs84.is_empty:
                folium.GeoJson(
                    pre_wgs84.__geo_interface__,
                    style_function=lambda x, tm=time_minutes: {
                        "fillColor": "#3b82f6" if tm == 5 else "#1d4ed8",
                        "color": "#1d4ed8",
                        "weight": 2,
                        "fillOpacity": 0.20 if tm == 5 else 0.14
                    },
                    tooltip=f"{facility_name}｜災前 {time_minutes} 分鐘"
                ).add_to(facility_group)

        if result["post_polygon"] is not None and not result["post_polygon"].is_empty:
            post_wgs84 = gpd.GeoSeries([result["post_polygon"]], crs="EPSG:3826").to_crs("EPSG:4326").iloc[0]
            post_wgs84 = simplify_for_web(post_wgs84, tolerance=0.00012)
            if post_wgs84 is not None and not post_wgs84.is_empty:
                folium.GeoJson(
                    post_wgs84.__geo_interface__,
                    style_function=lambda x, tm=time_minutes, rs=result: {
                        "fillColor": "#ef4444" if tm == 5 else "#b91c1c",
                        "color": "#b91c1c",
                        "weight": 2,
                        "fillOpacity": 0.20 if tm == 5 else 0.14
                    },
                    tooltip=f"{facility_name}｜災後 {time_minutes} 分鐘｜縮減 {result['shrinkage_percent']:.1f}%"
                ).add_to(facility_group)

    facility_group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

legend_html = """
<div style="
    position: fixed;
    bottom: 40px; left: 40px;
    width: 240px;
    background-color: white;
    border: 2px solid gray;
    z-index: 9999;
    font-size: 13px;
    padding: 10px;
    border-radius: 6px;
">
<b>Legend</b><br>
<div style="margin-top: 6px;">
  <span style="display:inline-block;width:16px;height:10px;background:#3b82f6;opacity:0.5;"></span> 災前 5 分鐘
</div>
<div>
  <span style="display:inline-block;width:16px;height:10px;background:#1d4ed8;opacity:0.4;"></span> 災前 10 分鐘
</div>
<div>
  <span style="display:inline-block;width:16px;height:10px;background:#ef4444;opacity:0.5;"></span> 災後 5 分鐘
</div>
<div>
  <span style="display:inline-block;width:16px;height:10px;background:#b91c1c;opacity:0.4;"></span> 災後 10 分鐘
</div>
<div>
  <span style="display:inline-block;width:16px;height:2px;background:gray;"></span> 道路網
</div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

map_file = OUTPUTS_DIR / "accessibility_interactive_map.html"
m.save(str(map_file))

print(f"✅ Saved static plots and interactive map to {OUTPUTS_DIR}")
m


## Step 10 — Optional AI Strategy Briefing

This cell is safe to skip. It only runs if a `GOOGLE_API_KEY` exists in your environment.

In [ ]:
print("=== Captain's Log: AI Strategy Briefing ===")

ai_available = False
try:
    import google.generativeai as genai
    ai_available = True
except ImportError:
    print("google-generativeai not installed. Bonus cell will be skipped.")

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    print("GOOGLE_API_KEY not found. Bonus cell will be skipped.")
    ai_available = False

if ai_available:
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel("gemini-1.5-flash")

    top5_info = top5_bottlenecks.to_dict("records")
    isolated_shelters = isolated_facilities
    prompt = f"""You are a transportation advisor at Hualien County Disaster Prevention Command Center.
Below are the road network analysis results from Typhoon Fung-wong:
Top 5 Bottleneck Nodes: {top5_info}
Accessibility Impact Table: {accessibility_table.to_dict('records')}
Isolated Facilities: {isolated_shelters}

In professional disaster-prevention language, please provide:
1. Priority road segments to clear (with reasoning)
2. Alternative rescue methods for isolated areas
3. Resource allocation recommendations
"""
    response = model.generate_content(prompt)
    ai_text = response.text
    with open(OUTPUTS_DIR / "ai_strategy_briefing.txt", "w", encoding="utf-8") as f:
        f.write(ai_text)
    print(ai_text[:3000])
    print("✅ AI briefing saved")


## Step 11 — README, Requirements, and Environment Template

In [ ]:
print("=== Captain's Log: Professional Documentation ===")

readme = f"""# ARIA v4.0 — Clean Rebuild

## Overview
This repository contains a cleaned notebook version of the Week 7 ARIA v4.0 assignment.  
The workflow analyzes how disaster rainfall alters road-network accessibility in **Hualien City, Taiwan**.

## Main Inputs
- `shelters_composite_risk.csv`
- `fungwong_202511.json`
- `hualien_network.graphml` (generated on first successful OSM fetch)

## Main Outputs
- `betweenness_centrality.csv`
- `top5_bottlenecks.csv`
- `bottleneck_terrain_overlay.csv`
- `rainfall_stations_filtered.geojson`
- `road_edge_midpoints_with_rainfall.geojson`
- `accessibility_impact_table.csv`
- `isochrone_detailed_results.csv`
- `isochrone_*.png`
- `accessibility_summary_plots.png`
- `accessibility_interactive_map.html`

## Workflow
1. Fetch or load Hualien road network
2. Project network to EPSG:3826
3. Estimate road speed and baseline travel time
4. Compute betweenness centrality and top-5 bottlenecks
5. Load shelters and filter them to the study network area
6. Overlay bottlenecks with nearest shelter terrain context
7. Parse CWA rainfall JSON and assign rainfall to road segments
8. Adjust travel times using congestion factors
9. Compute 5-minute and 10-minute pre/post-disaster isochrones
10. Compare area shrinkage and create static / interactive outputs

## Diagnostics / Fixes Used
- **CRS mismatch**: all analysis layers are forced to EPSG:3826; Folium conversion happens only at export time.
- **Shelters outside study area**: shelters are clipped to the network convex hull plus buffer.
- **Facility plotting errors**: facility points are drawn using the snapped nearest network node, not raw lon/lat.
- **Messy maps**: static plots use local road windows; Folium isochrones are grouped by facility and hidden by default.
- **Missing road speeds**: a default speed plus highway-type mapping is used when OSM maxspeed is missing.

## Notes
- The notebook includes an optional AI bonus cell that only runs when `GOOGLE_API_KEY` is available.
- If OSM download fails, rerun later or keep a cached GraphML copy in the notebook folder.
"""

requirements = """pandas
numpy
geopandas
networkx
osmnx
folium
matplotlib
shapely
jupyter
python-dotenv
google-generativeai
"""

env_example = """# Optional configuration
GOOGLE_API_KEY=
"""

with open(OUTPUTS_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme)
with open(OUTPUTS_DIR / "requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)
with open(OUTPUTS_DIR / ".env.example", "w", encoding="utf-8") as f:
    f.write(env_example)

print("✅ README.md, requirements.txt, and .env.example saved")


## Final Check

Use this cell to confirm the key output files after you run the full notebook.

In [ ]:
expected = [
    "betweenness_centrality.csv",
    "top5_bottlenecks.csv",
    "bottleneck_terrain_overlay.csv",
    "rainfall_stations_filtered.geojson",
    "road_edge_midpoints_with_rainfall.geojson",
    "accessibility_impact_table.csv",
    "isochrone_detailed_results.csv",
    "accessibility_summary_plots.png",
    "accessibility_interactive_map.html",
    "README.md",
]
status = pd.DataFrame({
    "file": expected,
    "exists": [(OUTPUTS_DIR / f).exists() for f in expected]
})
print(status.to_string(index=False))
